# Gradient Boosting Benchmark — WTI Weekly Return Prediction
## Walk-Forward Validation

Uses `sklearn.ensemble.HistGradientBoostingRegressor` — no native library dependencies.

**Evaluation approach:** Expanding-window walk-forward validation.
- Train on all history up to each fold's start date (no look-ahead)
- Test on the next `STEP_WEEKS` quarters of unseen data
- Re-train from scratch each fold — simulates real deployment

This produces ~20 out-of-sample folds instead of a single 80/20 split,
giving a more statistically robust estimate of directional accuracy.

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)

# ── Walk-forward parameters ────────────────────────────────────────────────────
INITIAL_TRAIN_WEEKS  = 104   # ~2 years minimum before first OOS fold
STEP_WEEKS           = 13    # quarterly test windows (~1 quarter per fold)
CONFIDENCE_THRESHOLD = 1.0   # |pred| threshold for filtered strategy

# ── Model constants ────────────────────────────────────────────────────────────
VOL_WINDOW  = 8       # rolling window for vol-normalised target
DECAY_RATE  = 0.995   # exponential sample weight decay (matches GRU)

FEATURE_COLS = [
    # ── News features ─────────────────────────────────────────────────────────
    'headline_count_lag1',
    'sentiment_mean_lag1',
    'sentiment_std_lag1',
    'sentiment_min_lag1',
    'sentiment_max_lag1',
    'negative_ratio_lag1',
    'positive_ratio_lag1',
    'log_headline_count_lag1',
    'opec_event_day_lag1',
    'opec_decision_day_lag1',
    'opec_sentiment_lag1',
    'disruption_event_day_lag1',
    'disruption_intensity_lag1',
    # ── FinBERT probability features ──────────────────────────────────────────
    'finbert_positive_mean_lag1',
    'finbert_negative_mean_lag1',
    'finbert_neutral_mean_lag1',
    # ── Regime features ───────────────────────────────────────────────────────
    'realized_vol_lag1',
    'regime_0_lag1',
    'regime_1_lag1',
    'regime_2_lag1',
    'hmm_regime_0_lag1',
    'hmm_regime_1_lag1',
    'hmm_regime_2_lag1',
    # ── EIA macro features ────────────────────────────────────────────────────
    'inventory_chg_kb_lag1',
    'inventory_surprise_kb_lag1',
    'production_chg_kbd_lag1',
    'imports_chg_kbd_lag1',
    'refinery_util_pct_lag1',
    'refinery_util_chg_pct_lag1',
    # ── FRED macro features ───────────────────────────────────────────────────
    'usd_ret_pct_lag1',
    'usd_mom_4w_lag1',
    'spx_ret_pct_lag1',
    'yield_10y_chg_lag1',
    # ── Paper features: decay sentiment + intensity-weighted variance ──────
    'broad_cs_di_lag1',
    'opec_cs_di_lag1',
    'disruption_cs_di_lag1',
    'broad_csi_v_lag1',
    'opec_csi_v_lag1',
    'disruption_csi_v_lag1',
]
TARGET_COL = 'close_pct_change'

## 2. Load Data

In [ ]:
df = pd.read_parquet('../data/features/feature_matrix.parquet')
df = df.sort_values('date').reset_index(drop=True)

print(f'Rows: {len(df)}  |  Date range: {df["date"].min().date()} → {df["date"].max().date()}')
missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f'WARNING — missing columns: {missing}')
else:
    print(f'All {len(FEATURE_COLS)} feature columns present')

## 3. Feature Matrix & Vol-Normalised Target

In [ ]:
X = df[FEATURE_COLS].fillna(0).values.astype(np.float32)
target_raw = df[TARGET_COL].values.astype(np.float32)

# Vol-normalised target — rolling std, no look-ahead (each week only uses prior weeks)
rolling_vol = (
    df[TARGET_COL]
    .rolling(VOL_WINDOW, min_periods=4)
    .std()
    .clip(lower=0.5)
    .bfill()
    .values.astype(np.float32)
)
y     = np.clip(target_raw / rolling_vol, -10, 10)
dates = df['date'].values
vols  = rolling_vol

n_folds = (len(X) - INITIAL_TRAIN_WEEKS) // STEP_WEEKS
print(f'Dataset       : {len(X)} weeks  ({df["date"].min().date()} → {df["date"].max().date()})')
print(f'First OOS date: {pd.Timestamp(dates[INITIAL_TRAIN_WEEKS]).date()}  (after {INITIAL_TRAIN_WEEKS}-week min training)')
print(f'Expected folds: ~{n_folds}  (step={STEP_WEEKS} weeks each)')

## 4. Walk-Forward Training
Each fold trains an expanding-window model (all history to date) and predicts the next quarter.
Exponential sample weights ensure recent weeks matter most in each fold.

In [ ]:
fold_records = []
all_preds    = []
last_model   = None   # final fold model — used for feature importance

for fold, test_start in enumerate(range(INITIAL_TRAIN_WEEKS, len(X), STEP_WEEKS)):
    test_end = min(test_start + STEP_WEEKS, len(X))
    if test_end - test_start < 5:
        break   # skip tiny trailing fold

    X_tr, y_tr = X[:test_start], y[:test_start]
    X_te       = X[test_start:test_end]
    y_te       = y[test_start:test_end]
    act_raw    = target_raw[test_start:test_end]
    vols_te    = vols[test_start:test_end]
    dates_te   = dates[test_start:test_end]

    # Exponential sample weights — decay toward older observations
    n     = len(X_tr)
    raw_w = np.array([DECAY_RATE ** (n - 1 - i) for i in range(n)], dtype=np.float32)
    sw    = raw_w / raw_w.mean()

    model_fold = HistGradientBoostingRegressor(
        max_iter            = 1000,
        learning_rate       = 0.02,
        max_depth           = 3,
        l2_regularization   = 2.0,
        loss                = 'absolute_error',
        early_stopping      = True,
        validation_fraction = 0.1,
        n_iter_no_change    = 30,
        random_state        = 42,
    )
    model_fold.fit(X_tr, y_tr, sample_weight=sw)

    preds_pct = model_fold.predict(X_te) * vols_te

    mae_f    = mean_absolute_error(act_raw, preds_pct)
    dir_acc_f = np.mean(np.sign(preds_pct) == np.sign(act_raw))

    fold_records.append({
        'fold':         fold,
        'train_weeks':  n,
        'test_start':   pd.Timestamp(dates_te[0]).date(),
        'test_end':     pd.Timestamp(dates_te[-1]).date(),
        'n_test':       len(act_raw),
        'mae':          mae_f,
        'dir_acc':      dir_acc_f,
        'iterations':   model_fold.n_iter_,
    })

    for i in range(len(act_raw)):
        all_preds.append({
            'fold':   fold,
            'date':   pd.Timestamp(dates_te[i]),
            'actual': float(act_raw[i]),
            'pred':   float(preds_pct[i]),
        })

    last_model = model_fold
    print(f'Fold {fold:2d}  train={n:3d}w  '
          f'test={pd.Timestamp(dates_te[0]).date()} → {pd.Timestamp(dates_te[-1]).date()}  '
          f'dir_acc={dir_acc_f:.1%}  iters={model_fold.n_iter_}')

folds_df  = pd.DataFrame(fold_records)
results   = pd.DataFrame(all_preds).sort_values('date').reset_index(drop=True)
print(f'\nTotal OOS weeks: {len(results)}  across {len(folds_df)} folds')

## 5. Evaluation — vs GRU Baseline

In [ ]:
# ── Aggregate metrics across all OOS folds ────────────────────────────────────
actual_all = results['actual'].values
pred_all   = results['pred'].values

mae_all  = mean_absolute_error(actual_all, pred_all)
rmse_all = np.sqrt(mean_squared_error(actual_all, pred_all))
dir_all  = np.mean(np.sign(pred_all) == np.sign(actual_all))

print('Walk-Forward OOS Summary')
print('=' * 52)
print(f'Total OOS weeks      : {len(results)}  across {len(folds_df)} folds')
print(f'Date range           : {results["date"].min().date()} → {results["date"].max().date()}')
print()
print(f'MAE (all folds)      : {mae_all:.4f}%')
print(f'RMSE (all folds)     : {rmse_all:.4f}%')
print(f'Dir Acc (all folds)  : {dir_all:.1%}')
print(f'Variance ratio       : {pred_all.std() / actual_all.std():.3f}')
print('=' * 52)

# ── Confidence-threshold breakdown (pooled OOS) ───────────────────────────────
print(f'\n{"Threshold":>12}  {"N kept":>6}  {"Dir Acc":>8}  {"Long":>6}  {"Short":>6}')
print('-' * 52)
for thresh in [0.0, 0.5, 1.0, 1.5, 2.0]:
    mask   = np.abs(pred_all) >= thresh
    n_kept = mask.sum()
    if n_kept < 5:
        break
    da      = np.mean(np.sign(pred_all[mask]) == np.sign(actual_all[mask]))
    n_long  = (pred_all[mask] > 0).sum()
    n_short = (pred_all[mask] < 0).sum()
    print(f'{thresh:>11.1f}%  {n_kept:>6}  {da:>8.1%}  {n_long:>6}  {n_short:>6}')

# ── Per-fold metrics table & bar chart ───────────────────────────────────────
print('\nPer-fold directional accuracy:')
print(folds_df[["fold","test_start","test_end","n_test","mae","dir_acc","iterations"]]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['darkorange' if d >= 0.55 else 'steelblue' if d >= 0.50 else 'salmon'
          for d in folds_df['dir_acc']]
ax.bar(folds_df['fold'], folds_df['dir_acc'], color=colors, edgecolor='white')
ax.axhline(0.50, color='black', linewidth=1, linestyle='--', label='50% baseline')
ax.axhline(dir_all, color='red', linewidth=1.5, linestyle='-', label=f'Overall {dir_all:.1%}')
ax.set_xlabel('Fold')
ax.set_ylabel('Directional Accuracy')
ax.set_title('Walk-Forward Directional Accuracy per Fold')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()


## 6. Feature Importance (Permutation)

In [ ]:
# Permutation importance on last_model — scored against all OOS data (pooled)
X_oos   = np.vstack([X[i] for i in range(INITIAL_TRAIN_WEEKS, len(X))])
# vol-normalise OOS target for consistent scoring units
y_oos   = np.concatenate([y[i:i+1] for i in range(INITIAL_TRAIN_WEEKS, len(X))])
# trim to results length (excludes any dropped trailing fold)
n_oos = len(results)
X_oos, y_oos = X_oos[:n_oos], y_oos[:n_oos]

perm = permutation_importance(last_model, X_oos, y_oos, n_repeats=20,
                               random_state=42, scoring='neg_mean_absolute_error')

importance = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 10))
colors = ['steelblue' if any(x in f for x in
          ['inventory','production','imports','refinery','usd','spx','yield'])
          else 'darkorange' if any(x in f for x in ['regime','hmm','realized_vol'])
          else 'grey'
          for f in importance.index]
importance.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Permutation Feature Importance (pooled OOS, vol-normalised MAE)\nBlue=macro  Orange=regime  Grey=news')
ax.set_xlabel('Importance (higher = more important)')
ax.axvline(0, color='black', linewidth=0.5)
ax.legend(handles=[
    mpatches.Patch(color='steelblue',  label='EIA / FRED macro'),
    mpatches.Patch(color='darkorange', label='Regime'),
    mpatches.Patch(color='grey',       label='News sentiment'),
])
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
print(importance.tail(10).to_string())


## 7. Strategy Comparison

In [ ]:
test_df = results.copy()
test_df = test_df.sort_values('date').reset_index(drop=True)

test_df['signal_naive']    = np.where(test_df['pred'] > 0, 1, 0)
test_df['signal_filtered'] = np.where(
    test_df['pred'] >=  CONFIDENCE_THRESHOLD,  1,
    np.where(
    test_df['pred'] <= -CONFIDENCE_THRESHOLD, -1, 0))

test_df['return_naive']    = test_df['signal_naive']    * test_df['actual']
test_df['return_filtered'] = test_df['signal_filtered'] * test_df['actual']

test_df['cum_buyhold']  = (1 + test_df['actual']         / 100).cumprod()
test_df['cum_naive']    = (1 + test_df['return_naive']   / 100).cumprod()
test_df['cum_filtered'] = (1 + test_df['return_filtered'] / 100).cumprod()

n_trades = (test_df['signal_filtered'] != 0).sum()
n_long   = (test_df['signal_filtered'] ==  1).sum()
n_short  = (test_df['signal_filtered'] == -1).sum()
filt_mask = test_df['signal_filtered'] != 0
dir_filt  = np.mean(
    np.sign(test_df.loc[filt_mask, 'pred']) == np.sign(test_df.loc[filt_mask, 'actual'])
) if filt_mask.any() else float('nan')

print(f'Walk-Forward Strategy Summary  ({test_df["date"].min().date()} → {test_df["date"].max().date()})')
print(f'OOS weeks                 : {len(test_df)}')
print(f'Confidence threshold      : |pred| >= {CONFIDENCE_THRESHOLD}%')
print(f'Trades taken              : {n_trades} / {len(test_df)}  ({n_long} long, {n_short} short)')
print(f'Directional acc (filtered): {dir_filt:.1%}')
print()
print(f'Final cumulative return:')
print(f'  Buy & Hold : {test_df["cum_buyhold"].iloc[-1]:.3f}x')
print(f'  Naive      : {test_df["cum_naive"].iloc[-1]:.3f}x')
print(f'  Filtered   : {test_df["cum_filtered"].iloc[-1]:.3f}x')

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Add fold-boundary shading
for _, row in folds_df.iterrows():
    axes[0].axvspan(pd.Timestamp(row['test_start']), pd.Timestamp(row['test_end']),
                    alpha=0.04, color='grey')
    axes[1].axvspan(pd.Timestamp(row['test_start']), pd.Timestamp(row['test_end']),
                    alpha=0.04, color='grey')

axes[0].plot(test_df['date'], test_df['actual'], label='Actual',    color='steelblue')
axes[0].plot(test_df['date'], test_df['pred'],   label='Predicted', color='darkorange', linestyle='--')
axes[0].axhline(0, color='black', linewidth=0.5, linestyle=':')
axes[0].axhline( CONFIDENCE_THRESHOLD, color='green', linewidth=0.8, linestyle=':', alpha=0.7)
axes[0].axhline(-CONFIDENCE_THRESHOLD, color='red',   linewidth=0.8, linestyle=':', alpha=0.7)
axes[0].set_title('GBM Walk-Forward — Predicted vs Actual Weekly Return (%)')
axes[0].set_ylabel('% Change')
axes[0].legend()

axes[1].plot(test_df['date'], test_df['cum_buyhold'],  label='Buy & Hold',               color='steelblue', linestyle='--')
axes[1].plot(test_df['date'], test_df['cum_naive'],    label='Naive (long if pred > 0)', color='grey',      linestyle=':')
axes[1].plot(test_df['date'], test_df['cum_filtered'], label=f'Filtered (|pred|≥{CONFIDENCE_THRESHOLD}%)', color='darkorange')
axes[1].axhline(1, color='black', linewidth=0.5, linestyle=':')
axes[1].set_title('Cumulative Return: Walk-Forward GBM Strategy (full OOS period)')
axes[1].set_ylabel('Growth of $1')
axes[1].legend()

plt.tight_layout()
plt.show()
